# CircuitVision · Improve the trained electronics detector (round 2)

This is a **follow-up training experiment**. It loads the student-trained YOLO11n model already published in our GitHub repository, downloads and merges the two annotated datasets, gives extra training exposure to rare classes (**Power Module and LDR**), and fine-tunes the model with larger images and more epochs. It still needs **T4 GPU in Google Colab** and your own private Roboflow API key.

The first model achieved **test mAP@50 ≈ 0.3966** and performed poorly on LDR and Power Module. No training setting guarantees better accuracy. Evaluate the held-out test set and real electronics in your lab before replacing the website model.

**Privacy:** The notebook only downloads public datasets and the student model already published in your GitHub repository. Keep your Roboflow API key private. The notebook does not upload model files to GitHub automatically.

**Data note:** Oversampling *existing* rare examples cannot replace genuinely diverse photographs. After this second training, annotate more real Power Module, LDR and LED images with tight boxes and collect separate test images before making accuracy claims.


## 1 · Install the tools

Press **Run all** in the Runtime menu. The cells execute in order. The notebook will stop with an understandable error if a dataset can't be downloaded or has incorrect labels.

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "roboflow>=1.1" "PyYAML>=6" "onnx>=1.15" "onnxruntime>=1.18" "onnxslim>=0.1"
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND — select Runtime → Change runtime type → T4 GPU')
assert torch.cuda.is_available(), 'Please enable a GPU in Colab before training.'

## 2 · Download the two annotated datasets

When prompted, paste **your Roboflow API key** into the hidden input field and press Enter. It is used only to access Roboflow during this session and is not saved in the notebook. If the dataset provider restricts access, visit its Roboflow page and follow the download/fork instructions shown there.

In [ ]:
from getpass import getpass
from pathlib import Path
from roboflow import Roboflow
import os, gc

api_key = getpass('Roboflow API key (hidden): ').strip()
assert api_key, 'Please provide your own Roboflow API key.'
rf = Roboflow(api_key=api_key)
DATA_ROOT = Path('/content/circuitvision_sources')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def get_source(workspace, project, version, local_folder):
    dest = DATA_ROOT / local_folder
    existing = list(dest.rglob('data.yaml')) if dest.exists() else []
    if existing:
        print(f'Reusing {local_folder} dataset:', existing[0].parent)
        return existing[0].parent
    try:
        exported = rf.workspace(workspace).project(project).version(version).download('yolov11',location=str(dest))
    except Exception as exc:
        raise RuntimeError(f'Could not download {workspace}/{project}/{version}. Check your Roboflow account and project access. Original error: {exc}') from exc
    return Path(exported.location)

sources = {
    'eesob': get_source('android-yolo', 'eesob', 4, 'eesob_v4'),
    'components': get_source('vishnupriyas-workspace', 'electronics-components-sveck', 7, 'components_v7'),
}
del rf, api_key
print('Both datasets downloaded:', sources)

## 3 · Combine annotations properly

A number such as `0` may refer to different objects in different datasets. This cell maps each source label by its **class name** and writes new, consistent YOLO bounding-box annotations. Some Roboflow export annotations are segmentation **polygons** (longer than five numbers); their vertices are converted into enclosing rectangular detection boxes. It preserves existing train/validation/test splits, skips byte-identical duplicates across splits, and does not invent boxes.

**Quick training** is recommended for your first run (roughly a few thousand images). It checks that everything works without spending hours on a full run. For a larger final experiment, change `QUICK_RUN` to `False`, then rerun the notebook on a GPU. A full training run can take many hours and Colab may disconnect.

**Limitation:** Images from one dataset may contain other parts that were not labeled in that dataset. Combining two sources does not automatically annotate those missing objects. The final model still needs manual evaluation on your school's actual components.
**This notebook includes the corrected polygon-to-box conversion** used in your successful first training run.


In [ ]:
# Round 2: a larger but still Colab-free-tier-friendly dataset.
# Each split limit applies separately to each source dataset.
QUICK_RUN = False
MAX_PER_SPLIT = {'train': 3200, 'val': 350, 'test': 250}


In [ ]:
"""Merge Roboflow YOLO object-detection exports without class-ID collisions.

Designed for CircuitVision's EESOB + Electronics Components datasets.
No labels are fabricated; missing label files count as empty images.
"""
from __future__ import annotations

import hashlib
import math
import random
import re
import shutil
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import yaml

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
# Preserve notable electronics nomenclature in result labels.
DISPLAY = {
    'led': 'LED', 'ldr': 'LDR', 'led matrix': 'LED Matrix',
    'ic': 'IC', 'lcd': 'LCD', 'lcd oled': 'LCD OLED', 'rfid': 'RFID',
    'rtc': 'RTC', 'ir sensor': 'IR Sensor', 'ir receiver': 'IR Receiver',
    'esp32': 'ESP32', 'esp wifi': 'ESP WiFi',
    'arduino uno': 'Arduino Uno', 'arduino nano': 'Arduino Nano',
    'arduino mini': 'Arduino Mini', 'arduino mega': 'Arduino Mega',
    'battery 9v': 'Battery 9V', '9v battery': 'Battery 9V',
    'power module': 'Power Module', 'resistor': 'Resistor',
    'capacitor': 'Capacitor', 'potentiometer': 'Potentiometer',
    'prototype pcb': 'Prototype PCB', '7 segment': '7-segment',
}


def class_key(value: str) -> str:
    normalized = re.sub(r'[_\-]+', ' ', str(value).strip().lower())
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    if not normalized:
        raise ValueError('Found blank class name')
    return normalized


def display_name(key: str) -> str:
    return DISPLAY.get(key, key.title())


def read_class_names(folder: Path) -> list[str]:
    yaml_path = folder / 'data.yaml'
    if not yaml_path.is_file():
        matches = list(folder.rglob('data.yaml'))
        if len(matches) != 1:
            raise FileNotFoundError(f'Expected one data.yaml under {folder}; found {len(matches)}')
        yaml_path = matches[0]
    obj = yaml.safe_load(yaml_path.read_text(encoding='utf-8'))
    names = obj.get('names')
    if isinstance(names, dict):
        ids = sorted(int(k) for k in names)
        if ids != list(range(len(ids))):
            raise ValueError(f'Class IDs in {yaml_path} are not contiguous starting at zero')
        names = [str(names[i] if i in names else names[str(i)]) for i in ids]
    if not isinstance(names, list) or not names:
        raise ValueError(f'Invalid or missing class list in {yaml_path}')
    keys = [class_key(n) for n in names]
    if len(set(keys)) != len(keys):
        raise ValueError(f'Repeated normalized class names in {yaml_path}: {names}')
    return [str(n) for n in names]


def split_images(folder: Path, split: str) -> Path | None:
    labels = ('valid', 'val') if split == 'val' else (split,)
    for part in labels:
        dest = folder / part / 'images'
        if dest.is_dir():
            return dest
    # Some Roboflow packages have an extra top-level directory.
    matches = [p for p in folder.rglob('images') if p.parent.name in labels and p.is_dir()]
    if len(matches) == 1:
        return matches[0]
    if matches:
        raise ValueError(f'Ambiguous {split} image directories under {folder}')
    return None


@dataclass
class Sample:
    path: Path
    labels: Path
    source: str
    split: str
    local_names: list[str]


def read_rows(path: Path, class_count: int) -> list[tuple[int, list[float]]]:
    """Read YOLO detection boxes or segmentation polygons as detection boxes.

    Roboflow's YOLO11 exports can contain polygon rows:
       class x1 y1 x2 y2 x3 y3 ...
    For object DETECTION we derive the tightest axis-aligned enclosing box.
    We preserve real boxes; no annotations are invented.
    """
    if not path.exists():
        return []
    parsed = []
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split()
        num_fields = len(parts)
        if num_fields != 5 and not (num_fields >= 7 and num_fields % 2 == 1):
            raise ValueError(
                f'{path}:{line_number}: expected class + 4 box coordinates '
                f'or class + 3+ polygon (x,y) pairs, got {num_fields} values')
        try:
            raw_id = float(parts[0])
            coords = [float(x) for x in parts[1:]]
        except ValueError as exc:
            raise ValueError(f'{path}:{line_number}: nonnumeric annotation value') from exc
        if not math.isfinite(raw_id) or any(not math.isfinite(x) for x in coords):
            raise ValueError(f'{path}:{line_number}: non-finite annotation value')
        local_id = int(raw_id)
        if raw_id != local_id or not 0 <= local_id < class_count:
            raise ValueError(f'{path}:{line_number}: class {parts[0]} not in 0..{class_count-1}')
        if any(not 0 <= x <= 1 for x in coords):
            raise ValueError(f'{path}:{line_number}: invalid normalized coordinates {coords}')
        if num_fields == 5:
            box = coords
        else:
            xs, ys = coords[::2], coords[1::2]
            min_x, max_x = min(xs), max(xs)
            min_y, max_y = min(ys), max(ys)
            box = [(min_x + max_x) / 2, (min_y + max_y) / 2,
                   max_x - min_x, max_y - min_y]
        if box[2] <= 0 or box[3] <= 0:
            raise ValueError(f'{path}:{line_number}: zero-area bounding box {box}')
        parsed.append((local_id, box))
    return parsed


def source_samples(folder: Path, source_name: str):
    names = read_class_names(folder)
    result = {'train': [], 'val': [], 'test': []}
    for split in result:
        image_dir = split_images(folder, split)
        if image_dir is None:
            continue
        label_dir = image_dir.parent / 'labels'
        for img in sorted(image_dir.iterdir()):
            if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
                result[split].append(Sample(img, label_dir / f'{img.stem}.txt', source_name, split, names))
    if not result['train'] or not result['val']:
        raise ValueError(f'{source_name}: both train and validation images are required')
    return result, names


def train_merge(roots: dict[str, Path], out: Path, limits: dict[str, int | None] | None = None,
                seed: int = 42) -> dict:
    """Create combined YOLO dataset. Respect source train/val/test splits.

    limits controls an optional quick-training cap per source/split, but does not
    change which split an image belongs to. All box class IDs are remapped.
    """
    if out.exists():
        raise FileExistsError(f'Remove old combined directory before rerunning: {out}')
    rng = random.Random(seed)
    sources = []
    for name, root in roots.items():
        samples, names = source_samples(Path(root), name)
        sources.append((name, samples, names))
    keys = sorted({class_key(n) for _name, _samples, names in sources for n in names})
    key_id = {key: ix for ix, key in enumerate(keys)}
    names = [display_name(key) for key in keys]
    if len(set(names)) != len(names):
        raise ValueError('Normalized display class names have duplicate values')
    counts = Counter()
    images = Counter()
    skipped_duplicates = 0
    missing_labels = 0
    seen_hashes = set()
    # Reserve holdout images first, to avoid leaking identical files into training.
    for split in ('test', 'val', 'train'):
        for source_name, samples, local_names in sources:
            candidates = list(samples[split])
            rng.shuffle(candidates)
            limit = (limits or {}).get(split)
            if limit is not None:
                limit = int(limit)
                if limit < 1:
                    raise ValueError(f'Sample limit for {split} must be positive')
                if split == 'train':
                    # Small training samples should not accidentally omit rare,
                    # important lab components. Favor a handful of annotated
                    # examples for each priority class before random sampling.
                    priority = {'resistor', 'led', 'ldr', 'power module',
                                'capacitor', 'breadboard', 'potentiometer'}
                    by_class = {name: [] for name in priority}
                    for image in candidates:
                        present = {class_key(local_names[i]) for i, _ in
                                   read_rows(image.labels, len(local_names))}
                        for name in present.intersection(priority):
                            by_class[name].append(image)
                    priority_images = []
                    selected = set()
                    for name in sorted(priority):
                        for image in by_class[name][:12]:
                            if image.path not in selected:
                                priority_images.append(image)
                                selected.add(image.path)
                    candidates = priority_images[:limit] + [
                        image for image in candidates if image.path not in selected
                    ][:max(0, limit - len(priority_images))]
                else:
                    candidates = candidates[:limit]
            for image in candidates:
                digest = hashlib.sha256(image.path.read_bytes()).hexdigest()
                if digest in seen_hashes:
                    skipped_duplicates += 1
                    continue
                seen_hashes.add(digest)
                rows = read_rows(image.labels, len(local_names))
                if not image.labels.is_file():
                    missing_labels += 1
                # Stable source prefix and digest prevent filename collisions.
                filename = f'{source_name}_{digest[:16]}{image.path.suffix.lower()}'
                dest_img = out / split / 'images' / filename
                dest_lbl = out / split / 'labels' / (Path(filename).stem + '.txt')
                dest_img.parent.mkdir(parents=True, exist_ok=True)
                dest_lbl.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(image.path, dest_img)
                with dest_lbl.open('w', encoding='utf-8') as f:
                    for local_id, box in rows:
                        new_id = key_id[class_key(local_names[local_id])]
                        f.write(str(new_id) + ' ' + ' '.join(f'{x:.6f}' for x in box) + '\n')
                        counts[(split, names[new_id])] += 1
                images[(split, source_name)] += 1
    if not sum(c for (s,_),c in images.items() if s == 'train') or not sum(c for (s,_),c in images.items() if s=='val'):
        raise ValueError('Combined dataset missing train or validation split')
    yaml_data = {'path': str(out.resolve()), 'train': 'train/images', 'val': 'val/images',
                 'test': 'test/images' if sum(c for (s,_),c in images.items() if s=='test') else 'val/images',
                 'nc': len(names), 'names': names}
    (out / 'data.yaml').write_text(yaml.safe_dump(yaml_data, sort_keys=False, allow_unicode=True), encoding='utf-8')
    return {'names': names, 'image_counts': dict(images), 'box_counts': dict(counts),
            'duplicate_images_skipped': skipped_duplicates, 'images_without_label_files': missing_labels,
            'yaml_path': str(out / 'data.yaml')}

In [ ]:
import shutil
from pathlib import Path
COMBINED_ROOT = Path('/content/circuitvision_combined')
if COMBINED_ROOT.exists():
    shutil.rmtree(COMBINED_ROOT)
report = train_merge(sources, COMBINED_ROOT, limits=MAX_PER_SPLIT, seed=42)
print('Merged classes:', len(report['names']))
print('Image counts by original source:', report['image_counts'])
print('Byte-identical duplicate images excluded:', report['duplicate_images_skipped'])
print('Images with missing label files (treated as backgrounds):', report['images_without_label_files'])
print('Merged configuration:', report['yaml_path'])
print('Training examples before rare-class oversampling:', sum(v for (split,source),v in report['image_counts'].items() if split=='train'))


## 4 · Check the important electronics classes

Make sure the combined dataset actually contains training boxes for the parts we care about. A class being **listed** does not guarantee high-quality detection; this is a basic annotation audit.

In [ ]:
from collections import Counter
important = ['Resistor','LED','LDR','Power Module','Capacitor','Breadboard','Potentiometer','Arduino Uno']
training_boxes = Counter({name: count for (split, name), count in report['box_counts'].items() if split == 'train'})
for label in important:
    print(f'{label:<18} {training_boxes.get(label, 0):>6} training boxes')
missing = [label for label in ['Resistor','LED','LDR','Power Module'] if training_boxes.get(label, 0)==0]
assert not missing, f'No training boxes found for {missing}. Check source dataset versions or increase sample size.'
print('✓ Priority classes have bounding-box annotations.')

## 5 · Start from your model and rebalance rare components

This section downloads the original `CircuitVision_Trained_Model.zip` from your *own public GitHub repository* and verifies its hash. It extracts `best.pt` **only to start training**, so the model already knows all 57 electronics classes.

Next, it gives extra training exposure to images containing **Power Module** and **LDR**. Extra copies stay entirely in the **training split** and Ultralytics applies randomized geometric/color augmentation during training. Validation and test splits stay unchanged, so the evaluation is honest.

This is a training experiment, **not a promise** of higher accuracy. Duplicate images do not create new viewpoints or fix mislabeled boxes. Collecting and annotating photos of your own components is still the most useful improvement.


In [ ]:
from pathlib import Path
from urllib.request import urlopen, Request
from collections import Counter
import hashlib, zipfile, shutil, json, io

BASELINE_ZIP_URL = (
    'https://raw.githubusercontent.com/'
    'aquinogeraldedmiranda-art/CircuitVision-YOLO/main/models/CircuitVision_Trained_Model.zip'
)
BASELINE_ZIP_SHA256 = '89e601fb299342bfedca6c48addfb04eaaf50ac5831d56c602a85f2cbc4b6d1e'
WEIGHTS_DIR = Path('/content/circuitvision_baseline')
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_PT = WEIGHTS_DIR / 'best.pt'

if not BASELINE_PT.exists():
    with urlopen(Request(BASELINE_ZIP_URL, headers={'User-Agent': 'CircuitVision-Training'}), timeout=120) as response:
        zip_bytes = response.read(18_000_000)
    assert hashlib.sha256(zip_bytes).hexdigest() == BASELINE_ZIP_SHA256, 'Unexpected baseline ZIP; stop to prevent training the wrong model'
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
        assert 'best.pt' in archive.namelist(), 'Baseline model ZIP has no best.pt'
        BASELINE_PT.write_bytes(archive.read('best.pt'))

# Oversample rare-class images only in train; no new boxes or fake labels.
train_images = COMBINED_ROOT / 'train' / 'images'
train_labels = COMBINED_ROOT / 'train' / 'labels'
assert train_images.exists() and train_labels.exists()
priority_copies = {'Power Module': 5, 'LDR': 2}
priority_class_ids = {name: report['names'].index(name) for name in priority_copies}
repeat_counts = Counter()
base_label_files = sorted(train_labels.glob('*.txt'))
for annotation in base_label_files:
    records = [ln.split() for ln in annotation.read_text().splitlines() if ln.strip()]
    present = {int(row[0]) for row in records if len(row) == 5}
    boost = max([priority_copies[name] for name, idx in priority_class_ids.items() if idx in present] or [0])
    if boost == 0:
        continue
    image = next((f for f in train_images.glob(annotation.stem+'.*') if f.is_file()), None)
    if image is None:
        raise FileNotFoundError(f'Missing image for {annotation}')
    for j in range(boost):
        suffix = f'_rare_boost_{j+1}'
        dest_img = train_images / (image.stem + suffix + image.suffix)
        dest_lbl = train_labels / (annotation.stem + suffix + '.txt')
        shutil.copy2(image, dest_img)
        shutil.copy2(annotation, dest_lbl)
        repeat_counts['extra_images'] += 1
    for name, idx in priority_class_ids.items():
        if idx in present:
            repeat_counts[name + ' source images'] += 1

print('Baseline model:', BASELINE_PT)
print('Oversampling report:', dict(repeat_counts))
print('CAUTION: Repeat training images are not independent new photos. Validate on held-out real pictures.')


## 6 · Fine-tune YOLO11 with 640-pixel images

This stage continues training your own 57-class YOLO11n from the baseline weights, instead of starting from a generic COCO detector. It trains for **up to 45 epochs** (early stopping may finish sooner). The test set is *not* used to choose model weights. A T4 GPU and a stable Colab session are needed; long training may disconnect on free Colab.

Built-in random augmentation includes scale, color, slight rotation and mosaics. The Power Module and LDR images are also repeated in the training set. **More varied annotated photos are still necessary** if those two classes remain weak.


In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Continue fine-tuning the actual CircuitVision YOLO11n 57-class checkpoint.
model = YOLO(str(BASELINE_PT))
existing_names = [str(v) for _, v in sorted(model.names.items())]
assert existing_names == report['names'], (
    'The two dataset class mappings changed. Do NOT fine-tune mismatched class indices.'
)
epochs = 45
image_size = 640
model.train(
    data=report['yaml_path'], epochs=epochs, imgsz=image_size,
    batch=8, device=0, workers=2, patience=15, seed=43,
    project='/content/circuitvision_runs', name='accuracy_round2', exist_ok=True,
    amp=True, plots=True, close_mosaic=10,
    mosaic=0.75, mixup=0.05, degrees=8.0,
    translate=0.10, scale=0.35, fliplr=0.50,
    save_period=5
)
BEST = Path('/content/circuitvision_runs/accuracy_round2/weights/best.pt')
assert BEST.is_file(), 'Training did not produce best.pt. Check the training logs above.'
print('Round 2 trained model:', BEST)


## 7 · Evaluate and export **before** updating the website

Measure aggregate test mAP@50 and mAP@50–95, and **individual class AP@50** for Resistor, LED, LDR, Power Module and Diode. If the new results do not improve on the baseline (~0.3966 mAP@50), we should keep the old deployment. Even if aggregate scores improve, poor Power Module/LDR performance means we need additional real annotated images.

Output is `CircuitVision_Accuracy_Round2.zip` containing the best weights, ONNX export, labels and dataset attribution. Training does not deploy automatically.


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import json, shutil, zipfile
trained = YOLO(str(BEST))
metrics = trained.val(data=report['yaml_path'], split='test', imgsz=image_size, device=0, plots=False)
print('Measured test mAP50:', round(float(metrics.box.map50), 4))
print('Measured test mAP50-95:', round(float(metrics.box.map), 4))
baseline_map50 = 0.3966
if float(metrics.box.map50) > baseline_map50:
    print('Held-out aggregate mAP50 exceeds round 1 — still inspect each important class.')
else:
    print('WARNING: Aggregate held-out mAP50 did not improve over round 1. Keep old model until inspected.')
# Per-class scores, important where Power Module/LDR have few test objects
for label in ['Resistor','LED','LDR','Power Module','Diode']:
    ci = report['names'].index(label)
    aps = [float(metrics.box.ap50[j]) for j, class_index in enumerate(metrics.box.ap_class_index) if int(class_index)==ci]
    print(f'{label:14} test AP50: '+ (f'{aps[0]:.3f}' if aps else 'No evaluable test matches'))

onnx_path = Path(trained.export(format='onnx', imgsz=640, opset=17,
                                dynamic=False, nms=False, simplify=True))
result_dir=Path('/content/CircuitVision_Trained_Model')
result_dir.mkdir(exist_ok=True)
shutil.copy2(BEST,result_dir/'best.pt')
shutil.copy2(onnx_path,result_dir/'best.onnx')
class_map={str(i): name for i,name in enumerate(report['names'])}
(result_dir/'classes.json').write_text(json.dumps(class_map, indent=2),encoding='utf-8')
(result_dir/'DATASET_SOURCES.txt').write_text(
    'EESOB: https://universe.roboflow.com/android-yolo/eesob/dataset/4 (CC BY 4.0)\n'
    'Electronics Components: https://universe.roboflow.com/vishnupriyas-workspace/electronics-components-sveck/dataset/7 (CC BY 4.0)\n'
    'Model: round-2 fine-tuning from CircuitVision student-trained YOLO11n using the datasets above.\n'
    'Do not report any evaluation values that were not actually measured.\n'
)
zip_path=Path('/content/CircuitVision_Accuracy_Round2.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for file in result_dir.iterdir(): z.write(file,arcname=file.name)
print('SUCCESS: trained model package ready:',zip_path)
print('Contents:', [f.name for f in result_dir.iterdir()])

## 8 · Download the round-2 checkpoint

The notebook creates **CircuitVision_Accuracy_Round2.zip**. Download it from the Colab Files pane, or run the cell below. Upload this ZIP into the CircuitVision ChatGPT conversation so we can verify the measurements before deploying. **Do not upload private Roboflow API keys.**


In [ ]:
from google.colab import files
files.download('/content/CircuitVision_Accuracy_Round2.zip')